In [ ]:
from ase.io import read
import pandas as pd
from collections import Counter
import os

# isolated atom energies
E0s = {
    3: -190.7590256408,   # Li
    8: -442.9888796243,   # O
    40: -1380.1817128081, # Zr
    57: -958.0774205521   # La
}

files = [
    "/home/mehuldarak/athena/dataset_trial0.extxyz",
    "/home/mehuldarak/athena/val_trial0.extxyz"
]

rows = []

for f in files:
    atoms_list = read(f, ":")

    for atoms in atoms_list:
        filename = atoms.info["filename"]
        ref_energy = atoms.info["REF_energy"]

        counts = Counter(atoms.get_atomic_numbers())
        e0_sum = sum(E0s[z] * n for z, n in counts.items())

        cohesive_energy = ref_energy - e0_sum

        rows.append({
            "Filename": filename,
            "REF_energy": ref_energy,
            "Cohesive_energy": cohesive_energy
        })

df = pd.DataFrame(rows)


                                            Filename     REF_energy  \
0  completed_Li_100_slab__LLZO_011_La_code71_sto_... -180180.184195   
1  completed_Li_100_slab__LLZO_010_La_order0_off_... -170953.693311   
2  completed_Li_100_slab__LLZO_001_Zr_code93_sto_... -180163.350910   
3  completed_Li_100_slab__LLZO_110_Li_order17_off... -169315.832501   
4  completed_Li_100_slab__LLZO_010_La_order0_off_... -170949.586718   
5  completed_Li_110_slab__LLZO_001_Zr_code93_sto_... -337213.826807   
6  completed_Li_100_slab__LLZO_010_La_order0_off_... -170974.490933   
7  completed_Li_100_slab__LLZO_001_Zr_code93_sto_... -180111.126575   

   Cohesive_energy  
0     -2229.435026  
1     -2067.582029  
2     -2212.601741  
3     -2078.735082  
4     -2063.475436  
5     -4203.411545  
6     -2088.379652  
7     -2160.377405  


In [3]:
df

,Filename,REF_energy,Cohesive_energy
0,completed_Li_100_slab__LLZO_011_La_code71_sto_...,-180180.184195,-2229.435026
1,completed_Li_100_slab__LLZO_010_La_order0_off_...,-170953.693311,-2067.582029
2,completed_Li_100_slab__LLZO_001_Zr_code93_sto_...,-180163.350910,-2212.601741
3,completed_Li_100_slab__LLZO_110_Li_order17_off...,-169315.832501,-2078.735082
4,completed_Li_100_slab__LLZO_010_La_order0_off_...,-170949.586718,-2063.475436
5,completed_Li_110_slab__LLZO_001_Zr_code93_sto_...,-337213.826807,-4203.411545
6,completed_Li_100_slab__LLZO_010_La_order0_off_...,-170974.490933,-2088.379652
7,completed_Li_100_slab__LLZO_001_Zr_code93_sto_...,-180111.126575,-2160.377405


In [4]:
df.to_csv("DFTFE_trial0_cohesive_energies.csv", index=False)

print("Saved to cohesive_energies.csv")

Saved to cohesive_energies.csv


In [11]:

import numpy as np
import pandas as pd
import torch
from ase.io import read
from mace.calculators import MACECalculator

datasets = [
    "/home/mehuldarak/athena/dataset_trial0.extxyz",
    "/home/mehuldarak/athena/val_trial0.extxyz"
]

model_path = "/home/mehuldarak/MACE_models/universal_09072025/mace-omat-0-medium.model"

torch.set_default_dtype(torch.float32)

calc = MACECalculator(
    model_path=model_path,
    device="cuda",
    default_dtype="float32"
)

rows = []

for file in datasets:
    structures = read(file, ":")

    for atoms in structures:
        atoms.calc = calc

        pred_forces = atoms.get_forces()
        ref_forces = atoms.arrays["REF_forces"]

        # absolute force error
        diff = pred_forces - ref_forces
        atom_errors = np.linalg.norm(diff, axis=1)

        # one scalar per structure
        force_error = atom_errors.mean()

        rows.append({
            "source_file": atoms.info["filename"],
            "force_error": 1000*force_error
        })

df = pd.DataFrame(rows)

print(df.head())

df.to_csv("mace_force_errors_uni.csv", index=False)

/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/mace/calculators/mace.py:166: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)


Using head default out of ['default']
Default dtype float32 does not match model dtype float64, converting models to float32.
                                         source_file  force_error
0  completed_Li_100_slab__LLZO_011_La_code71_sto_...   118.467918
1  completed_Li_100_slab__LLZO_010_La_order0_off_...   188.975398
2  completed_Li_100_slab__LLZO_001_Zr_code93_sto_...    90.148864
3  completed_Li_100_slab__LLZO_110_Li_order17_off...   149.087038
4  completed_Li_100_slab__LLZO_010_La_order0_off_...   198.584638


In [12]:
import numpy as np
import pandas as pd
import torch
from ase.io import read
from mace.calculators import MACECalculator

datasets = [
    "/home/mehuldarak/athena/dataset_trial0.extxyz",
    "/home/mehuldarak/athena/val_trial0.extxyz"
]

model_path = "/home/mehuldarak/athena/mace_seed_1_trial0_compiled.model"

torch.set_default_dtype(torch.float32)

calc = MACECalculator(
    model_path=model_path,
    device="cuda",
    default_dtype="float32"
)

rows = []

for file in datasets:
    structures = read(file, ":")

    for atoms in structures:
        atoms.calc = calc

        pred_forces = atoms.get_forces()
        ref_forces = atoms.arrays["REF_forces"]

        # absolute force error
        diff = pred_forces - ref_forces
        atom_errors = np.linalg.norm(diff, axis=1)

        # one scalar per structure
        force_error = atom_errors.mean()

        rows.append({
            "source_file": atoms.info["filename"],
            "force_error": 1000*force_error
        })

df = pd.DataFrame(rows)

print(df.head())

df.to_csv("mace_force_errors_trial0.csv", index=False)

/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/mace/calculators/mace.py:166: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  torch.load(f=model_path, map_location=device)
/home/mehuldarak/anaconda3/envs/mace/lib/python3.13/site-packages/torch/serialization.py:1493: UserWarning: 'torch.load' received a zip file that looks like a TorchScript archive dispatching to 'torch.jit.load' (call 'torch.jit.load' directly to silence this warning)
  warnings.warn(


Using head Default out of ['Default']
                                         source_file  force_error
0  completed_Li_100_slab__LLZO_011_La_code71_sto_...    55.845614
1  completed_Li_100_slab__LLZO_010_La_order0_off_...    72.046597
2  completed_Li_100_slab__LLZO_001_Zr_code93_sto_...    75.945740
3  completed_Li_100_slab__LLZO_110_Li_order17_off...    65.929350
4  completed_Li_100_slab__LLZO_010_La_order0_off_...    70.050614


In [13]:
import pandas as pd
import numpy as np

files = {
    "trial_model": "mace_force_errors_trial0.csv",
    "universal_model": "mace_force_errors_uni.csv"
}

rows = []

for name, file in files.items():
    df = pd.read_csv(file)

    errors = df["force_error"].values

    rows.append({
        "model": name,
        "min_meV_per_A": errors.min(),
        "max_meV_per_A": errors.max(),
        "rmse_meV_per_A": np.sqrt(np.mean(errors**2))
    })

stats = pd.DataFrame(rows)

print(stats)

stats.to_csv("force_error_model_comparison.csv", index=False)

             model  min_meV_per_A  max_meV_per_A  rmse_meV_per_A
0      trial_model      55.845614     130.378167       84.867343
1  universal_model      90.148864     234.904902      173.005595
